In [1]:
import polars as pl
import numpy as np
from scipy import stats

In [2]:
# μ_d = ค่าเฉลี่ยของ (ประตูทีมเหย้า - ประตูทีมเยือน) ต่อนัด
# H0 : μ_d = 0   ทีมเหย้าไม่ได้ทำประตูมากกว่าทีมเยือน (เฉลี่ยเท่ากัน)
# H1 : μ_d > 0   ทีมเหย้าทำประตูมากกว่าทีมเยือน (one-sided)
alpha = 0.05

In [3]:
matches = pl.read_csv("../data/elo_matches.csv", try_parse_dates=True).with_columns(
    pl.col("Season").str.slice(0, 4).cast(pl.Int32).alias("year")
)

In [4]:
d = (
    matches.filter(
        pl.col("MatchDate") >= pl.col("MatchDate").min().over("Season") + pl.duration(days=28)
    )
    .drop_nulls(["elo_home", "elo_away"])
    .with_columns(
        diff_elo=pl.col("elo_home") - pl.col("elo_away"),
        diff_goal=pl.col("FullTimeHomeGoals") - pl.col("FullTimeAwayGoals"),
    )
)

In [5]:
d.head()

Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,HomeShots,AwayShots,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards,elo_home,elo_away,exp_home,score_home,year,diff_elo,diff_goal
str,date,str,str,i64,i64,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,i32,f64,i64
"""2000/01""",2000-09-16,"""Arsenal""","""Coventry""",2,1,"""H""",1,0,"""H""",18,6,13,2,8,4,5,13,1,2,0,0,1519.141023,1487.672188,0.635369,1.0,2000,31.468835,1
"""2000/01""",2000-09-16,"""Aston Villa""","""Bradford""",2,0,"""H""",1,0,"""H""",12,8,7,4,5,6,8,12,0,2,0,0,1499.415305,1485.583781,0.61154,1.0,2000,13.831524,2
"""2000/01""",2000-09-16,"""Charlton""","""Tottenham""",1,0,"""H""",1,0,"""H""",9,10,5,5,7,5,8,10,0,0,0,0,1487.780968,1516.677042,0.551772,1.0,2000,-28.896074,1
"""2000/01""",2000-09-16,"""Everton""","""Man United""",1,3,"""A""",0,3,"""A""",6,17,4,9,6,5,10,11,4,1,0,0,1505.207155,1542.933734,0.539169,0.0,2000,-37.726579,-2
"""2000/01""",2000-09-16,"""Leeds""","""Ipswich""",1,2,"""A""",1,1,"""D""",11,12,6,7,6,3,12,19,1,3,0,0,1512.508303,1475.455403,0.642783,0.0,2000,37.0529,-1


In [6]:
(d.shape)
print(matches.shape, d.shape, matches["elo_home"].null_count())

(9380, 27) (8469, 29) 0


In [7]:
home = d["FullTimeHomeGoals"].to_numpy()
away = d["FullTimeAwayGoals"].to_numpy()

In [8]:
t, p = stats.ttest_rel(home, away, alternative="greater")
print(f"home mean={home.mean():.3f}, away mean={away.mean():.3f}, diff={np.mean(home - away):.3f}")
print(f"t={t:.2f}, p={p:.3g}")
if p < alpha:
    print(f"สรุปผล (t-test): ปฏิเสธ H0 (p = {p:.3g} < {alpha}) - ทีมเหย้าทำประตูมากกว่าทีมเยือน")
else:
    print(
        f"สรุปผล (t-test): ไม่สามารถปฏิเสธ H0 (p = {p:.3g} ≥ {alpha}) - ยังไม่พบหลักฐานว่าทีมเหย้าทำประตูมากกว่าทีมเยือน"
    )

home mean=1.541, away mean=1.179, diff=0.362
t=18.35, p=4.39e-74
สรุปผล (t-test): ปฏิเสธ H0 (p = 4.39e-74 < 0.05) - ทีมเหย้าทำประตูมากกว่าทีมเยือน


In [9]:
# ทางเลือกที่ไม่ต้องสมมติว่า normal
w, p_w = stats.wilcoxon(home - away, alternative="greater")
print(f"Wilcoxon p={p_w:.3g}")
if p_w < alpha:
    print(f"สรุปผล (Wilcoxon): ปฏิเสธ H0 (p = {p_w:.3g} < {alpha})")
else:
    print(f"สรุปผล (Wilcoxon): ไม่สามารถปฏิเสธ H0 (p = {p_w:.3g} ≥ {alpha})")

Wilcoxon p=3.75e-75
สรุปผล (Wilcoxon): ปฏิเสธ H0 (p = 3.75e-75 < 0.05)
